In [1]:
"""
COMPLETE SQLITE DATABASE TUTORIAL WITH PYTHON
==============================================
This tutorial covers everything about SQLite databases using Python,
from basic operations to advanced queries and optimization.
"""

import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# ============================================================================
# PART 1: CREATE AND CONNECT TO DATABASE
# ============================================================================

print("=" * 80)
print("PART 1: CREATE AND CONNECT TO DATABASE")
print("=" * 80)

# Create a connection to SQLite database (creates file if doesn't exist)
conn = sqlite3.connect('company_database.db')
print("\n✓ Database connection established")

# Create a cursor object to execute SQL commands
cursor = conn.cursor()
print("✓ Cursor object created")

# Check SQLite version
cursor.execute("SELECT sqlite_version()")
print(f"✓ SQLite version: {cursor.fetchone()[0]}")

# ============================================================================
# PART 2: CREATE TABLES
# ============================================================================

print("\n" + "=" * 80)
print("PART 2: CREATE TABLES")
print("=" * 80)

# Drop tables if they exist (for clean start)
cursor.execute("DROP TABLE IF EXISTS employees")
cursor.execute("DROP TABLE IF EXISTS departments")
cursor.execute("DROP TABLE IF EXISTS projects")
cursor.execute("DROP TABLE IF EXISTS employee_projects")

# Create employees table
create_employees_table = """
CREATE TABLE IF NOT EXISTS employees (
    employee_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    age INTEGER CHECK(age >= 18 AND age <= 100),
    email TEXT UNIQUE,
    salary REAL CHECK(salary > 0),
    department TEXT,
    join_date TEXT,
    performance INTEGER,
    is_active INTEGER DEFAULT 1,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
"""
cursor.execute(create_employees_table)
print("\n✓ Employees table created")

# Create departments table
create_departments_table = """
CREATE TABLE IF NOT EXISTS departments (
    dept_id INTEGER PRIMARY KEY AUTOINCREMENT,
    dept_name TEXT UNIQUE NOT NULL,
    manager TEXT,
    budget REAL,
    location TEXT
)
"""
cursor.execute(create_departments_table)
print("✓ Departments table created")

# Create projects table
create_projects_table = """
CREATE TABLE IF NOT EXISTS projects (
    project_id INTEGER PRIMARY KEY AUTOINCREMENT,
    project_name TEXT NOT NULL,
    start_date TEXT,
    end_date TEXT,
    budget REAL,
    status TEXT CHECK(status IN ('Active', 'Completed', 'On Hold'))
)
"""
cursor.execute(create_projects_table)
print("✓ Projects table created")

# Create employee_projects junction table (many-to-many relationship)
create_employee_projects_table = """
CREATE TABLE IF NOT EXISTS employee_projects (
    emp_proj_id INTEGER PRIMARY KEY AUTOINCREMENT,
    employee_id INTEGER,
    project_id INTEGER,
    role TEXT,
    hours_allocated INTEGER,
    FOREIGN KEY (employee_id) REFERENCES employees(employee_id),
    FOREIGN KEY (project_id) REFERENCES projects(project_id)
)
"""
cursor.execute(create_employee_projects_table)
print("✓ Employee_Projects junction table created")

# Commit changes
conn.commit()
print("✓ All tables committed to database")

# ============================================================================
# PART 3: VIEWING TABLE STRUCTURE
# ============================================================================

print("\n" + "=" * 80)
print("PART 3: VIEWING TABLE STRUCTURE")
print("=" * 80)

# Get list of all tables
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = cursor.fetchall()
print("\nTables in database:")
for table in tables:
    print(f"  - {table[0]}")

# Get table schema
cursor.execute("PRAGMA table_info(employees)")
columns = cursor.fetchall()
print("\nEmployees table schema:")
print(f"{'Column':<20} {'Type':<10} {'Not Null':<10} {'Default':<15} {'Primary Key'}")
print("-" * 80)
for col in columns:
    print(f"{col[1]:<20} {col[2]:<10} {col[3]:<10} {str(col[4]):<15} {col[5]}")

# ============================================================================
# PART 4: PREPARE SAMPLE DATA (FROM PREVIOUS CLEANING TUTORIAL)
# ============================================================================

print("\n" + "=" * 80)
print("PART 4: PREPARE SAMPLE CLEANED DATA")
print("=" * 80)

# Create cleaned sample data
employees_data = {
    'name': ['John Doe', 'Jane Smith', 'Bob Johnson', 'Alice Wong', 'Charlie Brown', 
             'David Lee', 'Emma Wilson', 'Frank Miller', 'Grace Chen', 'Henry Davis'],
    'age': [25, 30, 35, 28, 40, 32, 27, 45, 29, 38],
    'email': ['john@company.com', 'jane@company.com', 'bob@company.com', 
              'alice@company.com', 'charlie@company.com', 'david@company.com',
              'emma@company.com', 'frank@company.com', 'grace@company.com', 'henry@company.com'],
    'salary': [50000, 60000, 75000, 55000, 65000, 70000, 52000, 80000, 58000, 72000],
    'department': ['Sales', 'Sales', 'Marketing', 'Sales', 'Marketing', 'HR', 'HR', 'IT', 'IT', 'Marketing'],
    'join_date': ['2020-01-15', '2019-06-20', '2021-03-10', '2020-08-25', '2018-12-01',
                  '2022-02-14', '2021-09-30', '2017-05-18', '2020-11-08', '2019-03-22'],
    'performance': [85, 92, 78, 88, 95, 82, 87, 91, 89, 84]
}

df_employees = pd.DataFrame(employees_data)
print("\n✓ Sample cleaned data prepared")
print(df_employees.head())

# ============================================================================
# PART 5: INSERT DATA - METHOD 1 (Single Records)
# ============================================================================

print("\n" + "=" * 80)
print("PART 5: INSERT DATA - METHOD 1 (SINGLE RECORDS)")
print("=" * 80)

# Insert single record using execute
insert_query = """
INSERT INTO employees (name, age, email, salary, department, join_date, performance)
VALUES (?, ?, ?, ?, ?, ?, ?)
"""

# Insert one record
single_record = ('Test Employee', 30, 'test@company.com', 55000, 'IT', '2023-01-01', 85)
cursor.execute(insert_query, single_record)
conn.commit()
print("\n✓ Single record inserted")
print(f"  Last inserted row ID: {cursor.lastrowid}")

# Delete test record
cursor.execute("DELETE FROM employees WHERE name = 'Test Employee'")
conn.commit()

# ============================================================================
# PART 6: INSERT DATA - METHOD 2 (Multiple Records)
# ============================================================================

print("\n" + "=" * 80)
print("PART 6: INSERT DATA - METHOD 2 (MULTIPLE RECORDS)")
print("=" * 80)

# Insert multiple records using executemany
records = [tuple(x) for x in df_employees.values]
cursor.executemany(insert_query, records)
conn.commit()
print(f"\n✓ {cursor.rowcount} records inserted successfully")

# ============================================================================
# PART 7: INSERT DATA - METHOD 3 (From Pandas DataFrame)
# ============================================================================

print("\n" + "=" * 80)
print("PART 7: INSERT DATA - METHOD 3 (FROM PANDAS)")
print("=" * 80)

# Insert departments data
departments_data = {
    'dept_name': ['Sales', 'Marketing', 'HR', 'IT', 'Finance'],
    'manager': ['John Doe', 'Bob Johnson', 'David Lee', 'Frank Miller', 'Sarah Johnson'],
    'budget': [500000, 400000, 300000, 600000, 450000],
    'location': ['New York', 'Los Angeles', 'Chicago', 'San Francisco', 'Boston']
}

df_departments = pd.DataFrame(departments_data)
df_departments.to_sql('departments', conn, if_exists='append', index=False)
print("\n✓ Departments data loaded from DataFrame")

# Insert projects data
projects_data = {
    'project_name': ['Website Redesign', 'Mobile App', 'Data Migration', 'Security Audit', 'Marketing Campaign'],
    'start_date': ['2024-01-01', '2024-02-15', '2024-03-01', '2024-01-15', '2024-04-01'],
    'end_date': ['2024-06-30', '2024-08-31', '2024-05-31', '2024-04-30', '2024-12-31'],
    'budget': [100000, 150000, 80000, 50000, 120000],
    'status': ['Active', 'Active', 'Completed', 'Completed', 'Active']
}

df_projects = pd.DataFrame(projects_data)
df_projects.to_sql('projects', conn, if_exists='append', index=False)
print("✓ Projects data loaded from DataFrame")

# Insert employee-project relationships
emp_proj_data = [
    (1, 1, 'Developer', 40), (2, 1, 'Lead', 30), (3, 5, 'Manager', 35),
    (4, 1, 'Developer', 40), (5, 5, 'Designer', 40), (8, 2, 'Lead', 30),
    (9, 2, 'Developer', 40), (8, 3, 'Consultant', 20)
]

cursor.executemany(
    "INSERT INTO employee_projects (employee_id, project_id, role, hours_allocated) VALUES (?, ?, ?, ?)",
    emp_proj_data
)
conn.commit()
print("✓ Employee-Project relationships created")

# ============================================================================
# PART 8: SELECT QUERIES - BASIC
# ============================================================================

print("\n" + "=" * 80)
print("PART 8: SELECT QUERIES - BASIC")
print("=" * 80)

# Select all records
print("\n1. SELECT ALL:")
cursor.execute("SELECT * FROM employees LIMIT 5")
results = cursor.fetchall()
for row in results:
    print(row)

# Select specific columns
print("\n2. SELECT SPECIFIC COLUMNS:")
cursor.execute("SELECT name, department, salary FROM employees LIMIT 5")
results = cursor.fetchall()
for row in results:
    print(f"{row[0]:<20} {row[1]:<15} ${row[2]:,.2f}")

# Select with WHERE clause
print("\n3. WHERE CLAUSE (Salary > 60000):")
cursor.execute("SELECT name, salary FROM employees WHERE salary > 60000")
results = cursor.fetchall()
for row in results:
    print(f"{row[0]:<20} ${row[1]:,.2f}")

# Multiple conditions
print("\n4. MULTIPLE CONDITIONS (Sales AND salary > 55000):")
cursor.execute("""
    SELECT name, department, salary 
    FROM employees 
    WHERE department = 'Sales' AND salary > 55000
""")
results = cursor.fetchall()
for row in results:
    print(f"{row[0]:<20} {row[1]:<15} ${row[2]:,.2f}")

# Using IN clause
print("\n5. IN CLAUSE (Department in Sales, IT):")
cursor.execute("""
    SELECT name, department 
    FROM employees 
    WHERE department IN ('Sales', 'IT')
""")
results = cursor.fetchall()
for row in results:
    print(f"{row[0]:<20} {row[1]}")

# ============================================================================
# PART 9: SELECT QUERIES - ADVANCED
# ============================================================================

print("\n" + "=" * 80)
print("PART 9: SELECT QUERIES - ADVANCED")
print("=" * 80)

# LIKE operator
print("\n6. LIKE OPERATOR (Names starting with 'J'):")
cursor.execute("SELECT name, email FROM employees WHERE name LIKE 'J%'")
results = cursor.fetchall()
for row in results:
    print(f"{row[0]:<20} {row[1]}")

# BETWEEN operator
print("\n7. BETWEEN OPERATOR (Age between 28 and 35):")
cursor.execute("SELECT name, age FROM employees WHERE age BETWEEN 28 AND 35")
results = cursor.fetchall()
for row in results:
    print(f"{row[0]:<20} Age: {row[1]}")

# ORDER BY
print("\n8. ORDER BY (Salary descending):")
cursor.execute("SELECT name, salary FROM employees ORDER BY salary DESC LIMIT 5")
results = cursor.fetchall()
for i, row in enumerate(results, 1):
    print(f"{i}. {row[0]:<20} ${row[1]:,.2f}")

# GROUP BY
print("\n9. GROUP BY (Average salary by department):")
cursor.execute("""
    SELECT department, COUNT(*) as count, AVG(salary) as avg_salary, MAX(salary) as max_salary
    FROM employees
    GROUP BY department
    ORDER BY avg_salary DESC
""")
results = cursor.fetchall()
print(f"{'Department':<15} {'Count':<10} {'Avg Salary':<15} {'Max Salary'}")
print("-" * 60)
for row in results:
    print(f"{row[0]:<15} {row[1]:<10} ${row[2]:,.2f}      ${row[3]:,.2f}")

# HAVING clause
print("\n10. HAVING CLAUSE (Departments with avg salary > 60000):")
cursor.execute("""
    SELECT department, AVG(salary) as avg_salary
    FROM employees
    GROUP BY department
    HAVING AVG(salary) > 60000
""")
results = cursor.fetchall()
for row in results:
    print(f"{row[0]:<15} ${row[1]:,.2f}")

# ============================================================================
# PART 10: AGGREGATE FUNCTIONS
# ============================================================================

print("\n" + "=" * 80)
print("PART 10: AGGREGATE FUNCTIONS")
print("=" * 80)

print("\n11. AGGREGATE FUNCTIONS:")

# COUNT
cursor.execute("SELECT COUNT(*) FROM employees")
print(f"Total Employees: {cursor.fetchone()[0]}")

# SUM
cursor.execute("SELECT SUM(salary) FROM employees")
print(f"Total Salary Budget: ${cursor.fetchone()[0]:,.2f}")

# AVG
cursor.execute("SELECT AVG(salary) FROM employees")
print(f"Average Salary: ${cursor.fetchone()[0]:,.2f}")

# MIN and MAX
cursor.execute("SELECT MIN(salary), MAX(salary) FROM employees")
min_sal, max_sal = cursor.fetchone()
print(f"Salary Range: ${min_sal:,.2f} - ${max_sal:,.2f}")

# Multiple aggregates
cursor.execute("""
    SELECT 
        COUNT(*) as total_employees,
        AVG(age) as avg_age,
        AVG(performance) as avg_performance,
        SUM(salary) as total_payroll
    FROM employees
""")
result = cursor.fetchone()
print(f"\nCompany Statistics:")
print(f"  Total Employees: {result[0]}")
print(f"  Average Age: {result[1]:.1f}")
print(f"  Average Performance: {result[2]:.1f}")
print(f"  Total Payroll: ${result[3]:,.2f}")

# ============================================================================
# PART 11: JOIN OPERATIONS
# ============================================================================

print("\n" + "=" * 80)
print("PART 11: JOIN OPERATIONS")
print("=" * 80)

# INNER JOIN
print("\n12. INNER JOIN (Employees with Departments):")
cursor.execute("""
    SELECT e.name, e.salary, d.dept_name, d.location
    FROM employees e
    INNER JOIN departments d ON e.department = d.dept_name
    LIMIT 5
""")
results = cursor.fetchall()
for row in results:
    print(f"{row[0]:<20} ${row[1]:>8,.0f}  {row[2]:<12} {row[3]}")

# LEFT JOIN
print("\n13. LEFT JOIN (All employees and their projects):")
cursor.execute("""
    SELECT e.name, p.project_name, ep.role
    FROM employees e
    LEFT JOIN employee_projects ep ON e.employee_id = ep.employee_id
    LEFT JOIN projects p ON ep.project_id = p.project_id
    LIMIT 10
""")
results = cursor.fetchall()
for row in results:
    project = row[1] if row[1] else "No Project"
    role = row[2] if row[2] else "N/A"
    print(f"{row[0]:<20} {project:<25} {role}")

# Multiple JOINs
print("\n14. MULTIPLE JOINS (Complete employee-project details):")
cursor.execute("""
    SELECT 
        e.name as employee,
        d.dept_name as department,
        p.project_name as project,
        ep.role,
        ep.hours_allocated,
        p.status
    FROM employees e
    JOIN departments d ON e.department = d.dept_name
    JOIN employee_projects ep ON e.employee_id = ep.employee_id
    JOIN projects p ON ep.project_id = p.project_id
    WHERE p.status = 'Active'
""")
results = cursor.fetchall()
print(f"{'Employee':<20} {'Dept':<12} {'Project':<25} {'Role':<12} {'Hours':<8} {'Status'}")
print("-" * 100)
for row in results:
    print(f"{row[0]:<20} {row[1]:<12} {row[2]:<25} {row[3]:<12} {row[4]:<8} {row[5]}")

# ============================================================================
# PART 12: SUBQUERIES
# ============================================================================

print("\n" + "=" * 80)
print("PART 12: SUBQUERIES")
print("=" * 80)

# Subquery in WHERE
print("\n15. SUBQUERY - Employees earning above average:")
cursor.execute("""
    SELECT name, salary
    FROM employees
    WHERE salary > (SELECT AVG(salary) FROM employees)
    ORDER BY salary DESC
""")
results = cursor.fetchall()
for row in results:
    print(f"{row[0]:<20} ${row[1]:,.2f}")

# Subquery with IN
print("\n16. SUBQUERY - Employees in high-budget departments:")
cursor.execute("""
    SELECT name, department, salary
    FROM employees
    WHERE department IN (
        SELECT dept_name FROM departments WHERE budget > 400000
    )
""")
results = cursor.fetchall()
for row in results:
    print(f"{row[0]:<20} {row[1]:<12} ${row[2]:,.2f}")

# Correlated subquery
print("\n17. CORRELATED SUBQUERY - Highest paid in each department:")
cursor.execute("""
    SELECT e1.name, e1.department, e1.salary
    FROM employees e1
    WHERE e1.salary = (
        SELECT MAX(e2.salary)
        FROM employees e2
        WHERE e2.department = e1.department
    )
""")
results = cursor.fetchall()
for row in results:
    print(f"{row[0]:<20} {row[1]:<12} ${row[2]:,.2f}")

# ============================================================================
# PART 13: UPDATE QUERIES
# ============================================================================

print("\n" + "=" * 80)
print("PART 13: UPDATE QUERIES")
print("=" * 80)

# Simple update
print("\n18. UPDATE - Give 10% raise to Sales department:")
cursor.execute("SELECT AVG(salary) FROM employees WHERE department = 'Sales'")
old_avg = cursor.fetchone()[0]

cursor.execute("""
    UPDATE employees
    SET salary = salary * 1.10
    WHERE department = 'Sales'
""")
conn.commit()

cursor.execute("SELECT AVG(salary) FROM employees WHERE department = 'Sales'")
new_avg = cursor.fetchone()[0]
print(f"  Old average: ${old_avg:,.2f}")
print(f"  New average: ${new_avg:,.2f}")
print(f"  Rows affected: {cursor.rowcount}")

# Update with JOIN (using subquery)
print("\n19. UPDATE - Increase performance for high performers:")
cursor.execute("""
    UPDATE employees
    SET performance = performance + 5
    WHERE performance >= 90
""")
conn.commit()
print(f"  Rows affected: {cursor.rowcount}")

# Conditional update
print("\n20. UPDATE - Mark long-tenured employees:")
cursor.execute("""
    UPDATE employees
    SET performance = CASE
        WHEN performance + 3 <= 100 THEN performance + 3
        ELSE 100
    END
    WHERE join_date < '2020-01-01'
""")
conn.commit()
print(f"  Rows affected: {cursor.rowcount}")

# ============================================================================
# PART 14: DELETE QUERIES
# ============================================================================

print("\n" + "=" * 80)
print("PART 14: DELETE QUERIES")
print("=" * 80)

# Insert test records
test_records = [
    ('Test User 1', 25, 'test1@company.com', 40000, 'Sales', '2024-01-01', 70),
    ('Test User 2', 26, 'test2@company.com', 41000, 'HR', '2024-01-02', 71)
]
cursor.executemany(insert_query, test_records)
conn.commit()

# Delete with condition
print("\n21. DELETE - Remove test users:")
cursor.execute("DELETE FROM employees WHERE name LIKE 'Test User%'")
conn.commit()
print(f"  Rows deleted: {cursor.rowcount}")

# Delete with subquery
print("\n22. DELETE - Remove inactive employees (demo, no actual deletion):")
cursor.execute("SELECT COUNT(*) FROM employees WHERE is_active = 0")
print(f"  Inactive employees found: {cursor.fetchone()[0]}")
# cursor.execute("DELETE FROM employees WHERE is_active = 0")

# ============================================================================
# PART 15: TRANSACTIONS
# ============================================================================

print("\n" + "=" * 80)
print("PART 15: TRANSACTIONS")
print("=" * 80)

print("\n23. TRANSACTION - Transfer employee between departments:")
try:
    # Start transaction (implicit with sqlite3)
    cursor.execute("BEGIN TRANSACTION")
    
    # Update employee department
    cursor.execute("""
        UPDATE employees
        SET department = 'IT'
        WHERE name = 'Alice Wong'
    """)
    
    # Log the change (simulated)
    print("  ✓ Employee department updated")
    
    # Commit transaction
    conn.commit()
    print("  ✓ Transaction committed successfully")
    
except sqlite3.Error as e:
    # Rollback on error
    conn.rollback()
    print(f"  ✗ Transaction rolled back: {e}")

# Verify the change
cursor.execute("SELECT name, department FROM employees WHERE name = 'Alice Wong'")
result = cursor.fetchone()
print(f"  Current department: {result[1]}")

# ============================================================================
# PART 16: INDEXES FOR PERFORMANCE
# ============================================================================

print("\n" + "=" * 80)
print("PART 16: INDEXES FOR PERFORMANCE")
print("=" * 80)

# Create indexes
print("\n24. CREATE INDEXES:")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_department ON employees(department)")
print("  ✓ Index on department created")

cursor.execute("CREATE INDEX IF NOT EXISTS idx_salary ON employees(salary)")
print("  ✓ Index on salary created")

cursor.execute("CREATE INDEX IF NOT EXISTS idx_join_date ON employees(join_date)")
print("  ✓ Index on join_date created")

# List all indexes
cursor.execute("""
    SELECT name, tbl_name 
    FROM sqlite_master 
    WHERE type = 'index' AND tbl_name = 'employees'
""")
indexes = cursor.fetchall()
print("\n25. INDEXES on employees table:")
for idx in indexes:
    print(f"  - {idx[0]} on table {idx[1]}")

# Drop index
cursor.execute("DROP INDEX IF EXISTS idx_salary")
print("\n  ✓ Index idx_salary dropped")

# ============================================================================
# PART 17: VIEWS
# ============================================================================

print("\n" + "=" * 80)
print("PART 17: VIEWS")
print("=" * 80)

# Create view
print("\n26. CREATE VIEW:")
cursor.execute("""
    CREATE VIEW IF NOT EXISTS high_performers AS
    SELECT 
        e.name,
        e.department,
        e.salary,
        e.performance,
        d.manager
    FROM employees e
    JOIN departments d ON e.department = d.dept_name
    WHERE e.performance >= 85
""")
print("  ✓ View 'high_performers' created")

# Query the view
print("\n27. QUERY VIEW:")
cursor.execute("SELECT * FROM high_performers LIMIT 5")
results = cursor.fetchall()
print(f"{'Name':<20} {'Department':<12} {'Salary':<12} {'Performance':<12} {'Manager'}")
print("-" * 80)
for row in results:
    print(f"{row[0]:<20} {row[1]:<12} ${row[2]:<10,.0f} {row[3]:<12} {row[4]}")

# List all views
cursor.execute("SELECT name FROM sqlite_master WHERE type = 'view'")
views = cursor.fetchall()
print("\n28. ALL VIEWS in database:")
for view in views:
    print(f"  - {view[0]}")

# ============================================================================
# PART 18: DATE AND TIME FUNCTIONS
# ============================================================================

print("\n" + "=" * 80)
print("PART 18: DATE AND TIME FUNCTIONS")
print("=" * 80)

print("\n29. DATE FUNCTIONS:")

# Current date/time
cursor.execute("SELECT datetime('now'), date('now'), time('now')")
result = cursor.fetchone()
print(f"  Current datetime: {result[0]}")
print(f"  Current date: {result[1]}")
print(f"  Current time: {result[2]}")

# Calculate tenure
cursor.execute("""
    SELECT 
        name,
        join_date,
        CAST((julianday('now') - julianday(join_date)) / 365.25 AS INTEGER) as years_of_service
    FROM employees
    ORDER BY years_of_service DESC
    LIMIT 5
""")
results = cursor.fetchall()
print("\n30. EMPLOYEE TENURE:")
for row in results:
    print(f"{row[0]:<20} Joined: {row[1]}  Tenure: {row[2]} years")

# Employees joined in specific year
cursor.execute("""
    SELECT name, join_date
    FROM employees
    WHERE strftime('%Y', join_date) = '2020'
""")
results = cursor.fetchall()
print("\n31. EMPLOYEES JOINED IN 2020:")
for row in results:
    print(f"  {row[0]:<20} {row[1]}")

# ============================================================================
# PART 19: STRING FUNCTIONS
# ============================================================================

print("\n" + "=" * 80)
print("PART 19: STRING FUNCTIONS")
print("=" * 80)

print("\n32. STRING FUNCTIONS:")

# UPPER, LOWER, LENGTH
cursor.execute("""
    SELECT 
        name,
        UPPER(name) as uppercase,
        LOWER(name) as lowercase,
        LENGTH(name) as name_length
    FROM employees
    LIMIT 3
""")
results = cursor.fetchall()
for row in results:
    print(f"  Original: {row[0]}")
    print(f"  Upper: {row[1]}")
    print(f"  Lower: {row[2]}")
    print(f"  Length: {row[3]}")
    print()

# SUBSTR
cursor.execute("""
    SELECT 
        name,
        SUBSTR(name, 1, 3) as first_3_chars,
        SUBSTR(email, 1, INSTR(email, '@') - 1) as email_username
    FROM employees
    LIMIT 3
""")
results = cursor.fetchall()
print("33. SUBSTRING EXTRACTION:")
for row in results:
    print(f"  {row[0]:<20} First 3: {row[1]:<5} Username: {row[2]}")

# REPLACE
cursor.execute("""
    SELECT 
        email,
        REPLACE(email, '@company.com', '@newcompany.com') as new_email
    FROM employees
    LIMIT 3
""")
results = cursor.fetchall()
print("\n34. STRING REPLACEMENT:")
for row in results:
    print(f"  Old: {row[0]}")
    print(f"  New: {row[1]}")

# ============================================================================
# PART 20: MATHEMATICAL FUNCTIONS
# ============================================================================

print("\n" + "=" * 80)
print("PART 20: MATHEMATICAL FUNCTIONS")
print("=" * 80)

print("\n35. MATHEMATICAL FUNCTIONS:")

# ROUND, ABS
cursor.execute("""
    SELECT 
        name,
        salary,
        ROUND(salary * 1.15, 2) as projected_salary,
        ROUND(salary * 0.25, 2) as quarterly_salary,
        ABS(salary - 65000) as salary_diff_from_median
    FROM employees
    LIMIT 5
""")
results = cursor.fetchall()
print(f"{'Name':<20} {'Salary':<12} {'Projected':<12} {'Quarterly':<12} {'Diff'}")
print("-" * 80)
for row in results:
    print(f"{row[0]:<20} ${row[1]:<10,.0f} ${row[2]:<10,.0f} ${row[3]:<10,.0f} ${row[4]:,.0f}")

# MIN, MAX in single row
cursor.execute("""
    SELECT 
        name,
        salary,
        performance,
        MAX(salary, 60000) as min_guaranteed_salary
    FROM employees
    LIMIT 5
""")
results = cursor.fetchall()
print("\n36. MIN/MAX FUNCTIONS:")
for row in results:
    print(f"  {row[0]:<20} Salary: ${row[1]:>7,.0f}  Guaranteed: ${row[3]:>7,.0f}")

# ============================================================================
# PART 21: CASE STATEMENTS
# ============================================================================

print("\n" + "=" * 80)
print("PART 21: CASE STATEMENTS")
print("=" * 80)

print("\n37. CASE STATEMENTS - Categorize employees:")
cursor.execute("""
    SELECT 
        name,
        salary,
        performance,
        CASE 
            WHEN salary < 55000 THEN 'Entry Level'
            WHEN salary < 70000 THEN 'Mid Level'
            ELSE 'Senior Level'
        END as salary_grade,
        CASE
            WHEN performance >= 90 THEN 'Excellent'
            WHEN performance >= 85 THEN 'Good'
            WHEN performance >= 80 THEN 'Average'
            ELSE 'Needs Improvement'
        END as performance_rating
    FROM employees
    ORDER BY salary DESC
    LIMIT 8
""")
results = cursor.fetchall()
print(f"{'Name':<20} {'Salary':<12} {'Perf':<6} {'Grade':<15} {'Rating'}")
print("-" * 80)
for row in results:
    print(f"{row[0]:<20} ${row[1]:<10,.0f} {row[2]:<6} {row[3]:<15} {row[4]}")

# ============================================================================
# PART 22: WINDOW FUNCTIONS (ADVANCED)
# ============================================================================

print("\n" + "=" * 80)
print("PART 22: WINDOW FUNCTIONS (ADVANCED)")
print("=" * 80)

# ROW_NUMBER
print("\n38. ROW_NUMBER - Rank employees by salary in each department:")
cursor.execute("""
    SELECT 
        name,
        department,
        salary,
        ROW_NUMBER() OVER (PARTITION BY department ORDER BY salary DESC) as dept_rank
    FROM employees
    ORDER BY department, dept_rank
""")
results = cursor.fetchall()
print(f"{'Name':<20} {'Department':<12} {'Salary':<12} {'Rank'}")
print("-" * 60)
for row in results:
    print(f"{row[0]:<20} {row[1]:<12} ${row[2]:<10,.0f} {row[3]}")

# RANK and DENSE_RANK
print("\n39. RANK vs DENSE_RANK:")
cursor.execute("""
    SELECT 
        name,
        performance,
        RANK() OVER (ORDER BY performance DESC) as rank,
        DENSE_RANK() OVER (ORDER BY performance DESC) as dense_rank
    FROM employees
    ORDER BY performance DESC
    LIMIT 8
""")
results = cursor.fetchall()
print(f"{'Name':<20} {'Performance':<15} {'Rank':<10} {'Dense Rank'}")
print("-" * 60)
for row in results:
    print(f"{row[0]:<20} {row[1]:<15} {row[2]:<10} {row[3]}")

# LAG and LEAD
print("\n40. LAG and LEAD - Compare with previous/next salary:")
cursor.execute("""
    SELECT 
        name,
        salary,
        LAG(salary) OVER (ORDER BY salary) as prev_salary,
        LEAD(salary) OVER (ORDER BY salary) as next_salary
    FROM employees
    ORDER BY salary
    LIMIT 6
""")
results = cursor.fetchall()
print(f"{'Name':<20} {'Salary':<12} {'Previous':<12} {'Next'}")
print("-" * 60)
for row in results:
    prev = f"${row[2]:,.0f}" if row[2] else "N/A"
    next_sal = f"${row[3]:,.0f}" if row[3] else "N/A"
    print(f"{row[0]:<20} ${row[1]:<10,.0f} {prev:<12} {next_sal}")

# Running total
print("\n41. Running total of salaries:")
cursor.execute("""
    SELECT 
        name,
        salary,
        SUM(salary) OVER (ORDER BY employee_id) as running_total
    FROM employees
    LIMIT 6
""")
results = cursor.fetchall()
print(f"{'Name':<20} {'Salary':<12} {'Running Total'}")
print("-" * 50)
for row in results:
    print(f"{row[0]:<20} ${row[1]:<10,.0f} ${row[2]:,.0f}")

# ============================================================================
# PART 23: COMMON TABLE EXPRESSIONS (CTE)
# ============================================================================

print("\n" + "=" * 80)
print("PART 23: COMMON TABLE EXPRESSIONS (CTE)")
print("=" * 80)

print("\n42. CTE - Department statistics:")
cursor.execute("""
    WITH dept_stats AS (
        SELECT 
            department,
            COUNT(*) as emp_count,
            AVG(salary) as avg_salary,
            MAX(salary) as max_salary
        FROM employees
        GROUP BY department
    )
    SELECT 
        department,
        emp_count,
        ROUND(avg_salary, 2) as avg_salary,
        max_salary
    FROM dept_stats
    WHERE emp_count > 1
    ORDER BY avg_salary DESC
""")
results = cursor.fetchall()
print(f"{'Department':<15} {'Count':<10} {'Avg Salary':<15} {'Max Salary'}")
print("-" * 60)
for row in results:
    print(f"{row[0]:<15} {row[1]:<10} ${row[2]:<13,.2f} ${row[3]:,.0f}")

# Multiple CTEs
print("\n43. MULTIPLE CTEs - Top performers in high-paying departments:")
cursor.execute("""
    WITH high_paying_depts AS (
        SELECT department
        FROM employees
        GROUP BY department
        HAVING AVG(salary) > 60000
    ),
    top_performers AS (
        SELECT employee_id, name, department, performance
        FROM employees
        WHERE performance >= 85
    )
    SELECT 
        tp.name,
        tp.department,
        e.salary,
        tp.performance
    FROM top_performers tp
    JOIN high_paying_depts hpd ON tp.department = hpd.department
    JOIN employees e ON tp.employee_id = e.employee_id
    ORDER BY tp.performance DESC
""")
results = cursor.fetchall()
print(f"{'Name':<20} {'Department':<12} {'Salary':<12} {'Performance'}")
print("-" * 60)
for row in results:
    print(f"{row[0]:<20} {row[1]:<12} ${row[2]:<10,.0f} {row[3]}")

# Recursive CTE (organizational hierarchy example)
print("\n44. RECURSIVE CTE - Number sequence:")
cursor.execute("""
    WITH RECURSIVE cnt(x) AS (
        SELECT 1
        UNION ALL
        SELECT x+1 FROM cnt
        WHERE x < 5
    )
    SELECT x FROM cnt
""")
results = cursor.fetchall()
print("Generated sequence:", [row[0] for row in results])

# ============================================================================
# PART 24: QUERY DATA TO PANDAS
# ============================================================================

print("\n" + "=" * 80)
print("PART 24: EXPORT QUERY RESULTS TO PANDAS")
print("=" * 80)

# Read SQL query into DataFrame
df_result = pd.read_sql_query("""
    SELECT 
        e.name,
        e.department,
        e.salary,
        e.performance,
        d.manager,
        d.budget
    FROM employees e
    JOIN departments d ON e.department = d.dept_name
    ORDER BY e.salary DESC
""", conn)

print("\n45. Query results as Pandas DataFrame:")
print(df_result.head())

# Get entire table as DataFrame
df_employees_full = pd.read_sql_query("SELECT * FROM employees", conn)
print(f"\n46. Full employees table shape: {df_employees_full.shape}")
print(f"Columns: {df_employees_full.columns.tolist()}")

# ============================================================================
# PART 25: BACKUP AND EXPORT
# ============================================================================

print("\n" + "=" * 80)
print("PART 25: BACKUP AND EXPORT")
print("=" * 80)

# Export table to CSV
df_employees_full.to_csv('employees_backup.csv', index=False)
print("\n47. ✓ Data exported to CSV: employees_backup.csv")

# Create backup database
backup_conn = sqlite3.connect('company_database_backup.db')
with backup_conn:
    conn.backup(backup_conn)
backup_conn.close()
print("48. ✓ Database backed up to: company_database_backup.db")

# ============================================================================
# PART 26: EXPLAIN QUERY PLAN
# ============================================================================

print("\n" + "=" * 80)
print("PART 26: QUERY OPTIMIZATION - EXPLAIN QUERY PLAN")
print("=" * 80)

print("\n49. EXPLAIN QUERY PLAN:")
cursor.execute("""
    EXPLAIN QUERY PLAN
    SELECT * FROM employees WHERE department = 'Sales' AND salary > 55000
""")
results = cursor.fetchall()
print("Query execution plan:")
for row in results:
    print(f"  {row}")

# ============================================================================
# PART 27: PRAGMA COMMANDS
# ============================================================================

print("\n" + "=" * 80)
print("PART 27: PRAGMA COMMANDS - DATABASE INFO")
print("=" * 80)

# Database info
cursor.execute("PRAGMA database_list")
print("\n50. Database list:")
for row in cursor.fetchall():
    print(f"  {row}")

# Table info
cursor.execute("PRAGMA table_info(employees)")
print("\n51. Table structure (employees):")
columns = cursor.fetchall()
for col in columns:
    print(f"  {col[1]}: {col[2]}")

# Foreign keys
cursor.execute("PRAGMA foreign_keys")
print(f"\n52. Foreign keys enabled: {cursor.fetchone()[0]}")

# Database size
cursor.execute("PRAGMA page_count")
page_count = cursor.fetchone()[0]
cursor.execute("PRAGMA page_size")
page_size = cursor.fetchone()[0]
db_size = (page_count * page_size) / 1024
print(f"53. Database size: {db_size:.2f} KB")

# ============================================================================
# PART 28: FULL-TEXT SEARCH
# ============================================================================

print("\n" + "=" * 80)
print("PART 28: FULL-TEXT SEARCH (FTS5)")
print("=" * 80)

# Create FTS table
cursor.execute("""
    CREATE VIRTUAL TABLE IF NOT EXISTS employees_fts 
    USING fts5(name, email, department)
""")

# Populate FTS table
cursor.execute("""
    INSERT INTO employees_fts (name, email, department)
    SELECT name, email, department FROM employees
""")
conn.commit()
print("\n54. ✓ Full-text search table created and populated")

# Perform full-text search
cursor.execute("""
    SELECT name, email, department 
    FROM employees_fts 
    WHERE employees_fts MATCH 'john OR alice'
""")
results = cursor.fetchall()
print("\n55. Full-text search results for 'john OR alice':")
for row in results:
    print(f"  {row[0]:<20} {row[1]:<25} {row[2]}")

# ============================================================================
# PART 29: CONDITIONAL INSERTS (UPSERT)
# ============================================================================

print("\n" + "=" * 80)
print("PART 29: UPSERT (INSERT OR REPLACE)")
print("=" * 80)

# Insert or replace
print("\n56. UPSERT operation:")
cursor.execute("""
    INSERT INTO employees (name, age, email, salary, department, join_date, performance)
    VALUES ('John Doe', 26, 'john@company.com', 52000, 'Sales', '2020-01-15', 86)
    ON CONFLICT(email) DO UPDATE SET
        age = excluded.age,
        salary = excluded.salary,
        performance = excluded.performance
""")
conn.commit()
print(f"  ✓ Upsert completed, rows affected: {cursor.rowcount}")

# Verify
cursor.execute("SELECT name, age, salary, performance FROM employees WHERE email = 'john@company.com'")
result = cursor.fetchone()
print(f"  Updated record: {result}")

# ============================================================================
# PART 30: ADVANCED ANALYTICS QUERIES
# ============================================================================

print("\n" + "=" * 80)
print("PART 30: ADVANCED ANALYTICS QUERIES")
print("=" * 80)

# Salary percentiles
print("\n57. Salary percentiles:")
cursor.execute("""
    SELECT 
        'P25' as percentile,
        (SELECT salary FROM employees ORDER BY salary LIMIT 1 OFFSET 
         (SELECT COUNT(*) FROM employees) * 25 / 100) as salary
    UNION ALL
    SELECT 
        'P50',
        (SELECT salary FROM employees ORDER BY salary LIMIT 1 OFFSET 
         (SELECT COUNT(*) FROM employees) * 50 / 100)
    UNION ALL
    SELECT 
        'P75',
        (SELECT salary FROM employees ORDER BY salary LIMIT 1 OFFSET 
         (SELECT COUNT(*) FROM employees) * 75 / 100)
""")
results = cursor.fetchall()
for row in results:
    print(f"  {row[0]}: ${row[1]:,.0f}")

# Department comparison
print("\n58. Department performance comparison:")
cursor.execute("""
    SELECT 
        department,
        COUNT(*) as employees,
        ROUND(AVG(salary), 2) as avg_salary,
        ROUND(AVG(performance), 2) as avg_performance,
        ROUND(AVG(salary) / AVG(performance), 2) as cost_per_performance_point
    FROM employees
    GROUP BY department
    ORDER BY avg_performance DESC
""")
results = cursor.fetchall()
print(f"{'Dept':<12} {'Count':<8} {'Avg Salary':<12} {'Avg Perf':<10} {'Cost/Point'}")
print("-" * 70)
for row in results:
    print(f"{row[0]:<12} {row[1]:<8} ${row[2]:<10,.0f} {row[3]:<10} ${row[4]:,.0f}")

# Cohort analysis by join year
print("\n59. Cohort analysis by join year:")
cursor.execute("""
    SELECT 
        strftime('%Y', join_date) as join_year,
        COUNT(*) as new_hires,
        ROUND(AVG(salary), 2) as avg_starting_salary,
        ROUND(AVG(performance), 2) as avg_performance
    FROM employees
    WHERE join_date IS NOT NULL
    GROUP BY join_year
    ORDER BY join_year
""")
results = cursor.fetchall()
print(f"{'Year':<8} {'New Hires':<12} {'Avg Salary':<15} {'Avg Performance'}")
print("-" * 60)
for row in results:
    print(f"{row[0]:<8} {row[1]:<12} ${row[2]:<13,.0f} {row[3]}")

# ============================================================================
# PART 31: CLEANUP AND CLOSE
# ============================================================================

print("\n" + "=" * 80)
print("PART 31: CLEANUP AND CLOSE CONNECTION")
print("=" * 80)

# Get final statistics
cursor.execute("SELECT COUNT(*) FROM employees")
total_employees = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(*) FROM departments")
total_departments = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(*) FROM projects")
total_projects = cursor.fetchone()[0]

print(f"\n60. Final database statistics:")
print(f"  Total Employees: {total_employees}")
print(f"  Total Departments: {total_departments}")
print(f"  Total Projects: {total_projects}")

# Close cursor and connection
cursor.close()
conn.close()
print("\n✓ Cursor closed")
print("✓ Database connection closed")

# ============================================================================
# SUMMARY AND BEST PRACTICES
# ============================================================================

print("\n" + "=" * 80)
print("TUTORIAL COMPLETE - SUMMARY")
print("=" * 80)

print("""
KEY CONCEPTS COVERED:
====================
1. Database Connection & Setup
2. Creating Tables with Constraints
3. Inserting Data (single, bulk, from pandas)
4. Basic SELECT Queries
5. Advanced SELECT (JOIN, SUBQUERY, CTE)
6. Aggregate Functions
7. UPDATE and DELETE Operations
8. Transactions
9. Indexes for Performance
10. Views
11. Date/Time Functions
12. String Functions
13. Mathematical Functions
14. CASE Statements
15. Window Functions
16. Common Table Expressions
17. Full-Text Search
18. UPSERT Operations
19. Query Optimization
20. Data Export/Backup

BEST PRACTICES:
===============
✓ Always use parameterized queries (?) to prevent SQL injection
✓ Commit transactions after data modifications
✓ Use indexes on frequently queried columns
✓ Close connections when done
✓ Use transactions for related operations
✓ Create views for complex, repeated queries
✓ Use CTEs for better query readability
✓ Regular backups of important databases
✓ Use EXPLAIN QUERY PLAN for optimization
✓ Properly handle NULL values

FILES CREATED:
==============
- company_database.db (main database)
- company_database_backup.db (backup)
- employees_backup.csv (CSV export)

NEXT STEPS:
===========
- Practice writing complex queries
- Learn about database normalization
- Explore SQLite performance tuning
- Integrate with web applications
- Study database security best practices
""")

PART 1: CREATE AND CONNECT TO DATABASE

✓ Database connection established
✓ Cursor object created
✓ SQLite version: 3.37.0

PART 2: CREATE TABLES

✓ Employees table created
✓ Departments table created
✓ Projects table created
✓ Employee_Projects junction table created
✓ All tables committed to database

PART 3: VIEWING TABLE STRUCTURE

Tables in database:
  - employees
  - sqlite_sequence
  - departments
  - projects
  - employee_projects

Employees table schema:
Column               Type       Not Null   Default         Primary Key
--------------------------------------------------------------------------------
employee_id          INTEGER    0          None            1
name                 TEXT       1          None            0
age                  INTEGER    0          None            0
email                TEXT       0          None            0
salary               REAL       0          None            0
department           TEXT       0          None            0
join_date     